In [ ]:
pip install ultralytics


In [ ]:

!pip install mediapipe -q
!pip install opencv-python pybullet ultralytics -q

!pip install protobuf==3.20.3 -q

print("y")

ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/commands/install.py", line 362, in run
    resolver = self.make_resolver(
        preparer=preparer,
    ...<8 lines>...
        use_pep517=options.use_pep517,
    )
  File "/usr/local/lib/python3.13/dist-packages/pip/_internal/cli/req_command.py", line 177, in make_resolver
    return pip._internal.resolution.resolvelib.resolver.Resolver(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        preparer=preparer,
        ^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
        py_version_info=py_version_info,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File

In [ ]:
!pip install opencv-python mediapipe pybullet ultralytics -q
!pip install protobuf==3.20.3 -q

In [ ]:
import cv2
import mediapipe as mp
import pybullet as p
import pybullet_data
import numpy as np
import math
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import PIL
import io

p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.resetDebugVisualizerCamera(cameraDistance=1.8, cameraYaw=35, cameraPitch=-20, cameraTargetPosition=[0, 0, 0.5])
p.setGravity(0, 0, -9.81)
p.loadURDF("plane.urdf")

SCALE = 1.2
SMOOTHING = 0.35

def create_capsule(radius, height, color):
    visual_shape = p.createVisualShape(shapeType=p.GEOM_CAPSULE, radius=radius, length=height, rgbaColor=color)
    collision_shape = p.createCollisionShape(shapeType=p.GEOM_CAPSULE, radius=radius, height=height)
    body_id = p.createMultiBody(baseMass=0, baseCollisionShapeIndex=collision_shape, baseVisualShapeIndex=visual_shape)
    return body_id

def create_sphere(radius, color):
    visual_shape = p.createVisualShape(shapeType=p.GEOM_SPHERE, radius=radius, rgbaColor=color)
    body_id = p.createMultiBody(baseMass=0, baseVisualShapeIndex=visual_shape)
    return body_id

HEAD = create_sphere(0.08, [0.9, 0.7, 0.1, 1])
TORSO = create_capsule(0.09, 0.25, [0.1, 0.5, 0.2, 1])
LEFT_UPPER_ARM = create_capsule(0.035, 0.18, [0.1, 0.2, 0.8, 1])
LEFT_FOREARM = create_capsule(0.03, 0.18, [0.8, 0.6, 0.1, 1])
RIGHT_UPPER_ARM = create_capsule(0.035, 0.18, [0.1, 0.2, 0.8, 1])
RIGHT_FOREARM = create_capsule(0.03, 0.18, [0.8, 0.6, 0.1, 1])
LEFT_THIGH = create_capsule(0.045, 0.22, [0.2, 0.6, 0.2, 1])
LEFT_SHIN = create_capsule(0.035, 0.22, [0.8, 0.7, 0.1, 1])
RIGHT_THIGH = create_capsule(0.045, 0.22, [0.2, 0.6, 0.2, 1])
RIGHT_SHIN = create_capsule(0.035, 0.22, [0.8, 0.7, 0.1, 1])
LEFT_SHOULDER_MARKER = create_sphere(0.045, [0.1, 0.1, 0.8, 1])
RIGHT_SHOULDER_MARKER = create_sphere(0.045, [0.1, 0.1, 0.8, 1])
LEFT_ELBOW_MARKER = create_sphere(0.04, [0.8, 0.1, 0.1, 1])
RIGHT_ELBOW_MARKER = create_sphere(0.04, [0.8, 0.1, 0.1, 1])
LEFT_HIP_MARKER = create_sphere(0.05, [0.2, 0.2, 0.8, 1])
RIGHT_HIP_MARKER = create_sphere(0.05, [0.2, 0.2, 0.8, 1])
LEFT_KNEE_MARKER = create_sphere(0.045, [0.8, 0.1, 0.1, 1])
RIGHT_KNEE_MARKER = create_sphere(0.045, [0.8, 0.1, 0.1, 1])

def set_body_between(body_id, start, end):
    start = np.array(start, dtype=np.float32)
    end = np.array(end, dtype=np.float32)
    direction = end - start
    length = np.linalg.norm(direction)
    if length < 0.001:
        return
    center = (start + end) / 2.0
    direction = direction / length
    default_axis = np.array([0.0, 0.0, 1.0])
    cross = np.cross(default_axis, direction)
    dot = np.dot(default_axis, direction)
    dot = np.clip(dot, -1.0, 1.0)
    angle = math.acos(dot)
    cross_norm = np.linalg.norm(cross)
    if cross_norm < 0.0001:
        quaternion = [0, 0, 0, 1]
    else:
        axis = cross / cross_norm
        quaternion = p.getQuaternionFromAxisAngle(axis.tolist(), angle)
    p.resetBasePositionAndOrientation(body_id, center.tolist(), quaternion)

previous_positions = {}

def smooth_position(name, position):
    position = np.array(position)
    if name not in previous_positions:
        previous_positions[name] = position
    previous_positions[name] = previous_positions[name] * (1.0 - SMOOTHING) + position * SMOOTHING
    return previous_positions[name]

def landmark_to_world(landmark):
    x = (landmark.x - 0.5) * SCALE
    y = -landmark.z * SCALE * 0.7
    z = (1.0 - landmark.y) * SCALE * 0.7
    return np.array([x, y, z])

def capture_pybullet_frame():
    width, height = 320, 240
    view_matrix = p.computeViewMatrixFromYawPitchRoll(cameraTargetPosition=[0, 0, 0.5], distance=1.8, yaw=35, pitch=-20, roll=0, upAxisIndex=2)
    proj_matrix = p.computeProjectionMatrixFOV(fov=60, aspect=float(width)/height, nearVal=0.1, farVal=100.0)
    _, _, rgb_img, _, _ = p.getCameraImage(width, height, viewMatrix=view_matrix, projectionMatrix=proj_matrix, renderer=p.ER_TINY_RENDERER)
    rgb_img = np.reshape(rgb_img, (height, width, 4))
    rgb_img = rgb_img[:, :, :3]
    rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2BGR)
    return rgb_img

In [ ]:
def init_webcam():
    js = Javascript('''
        async function init() {
            const video = document.createElement('video');
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            video.play();
            await new Promise(r => video.onloadedmetadata = r);
            window.video = video;
            return true;
        }
        async function captureFrame() {
            const canvas = document.createElement('canvas');
            canvas.width = window.video.videoWidth;
            canvas.height = window.video.videoHeight;
            canvas.getContext('2d').drawImage(window.video, 0, 0);
            return canvas.toDataURL('image/jpeg', 0.8);
        }
    ''')
    display(js)
    eval_js('init()')

init_webcam()

In [ ]:
import time
from IPython.display import clear_output

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

try:
    yolo_model = YOLO('yolov26.pt')
except Exception as e:
    yolo_model = YOLO('yolov8n.pt')

FRAMES_TO_RUN = 150

with mp_pose.Pose(static_image_mode=False, model_complexity=1, smooth_landmarks=True, enable_segmentation=False, min_detection_confidence=0.6, min_tracking_confidence=0.6) as pose:
    for i in range(FRAMES_TO_RUN):
        try:
            data_url = eval_js('captureFrame()')
            b64 = data_url.split(',')[1]
            nparr = np.frombuffer(b64decode(b64), np.uint8)
            frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
            if frame is None:
                continue
            frame = cv2.flip(frame, 1)

            yolo_results = yolo_model(frame, verbose=False)
            annotated_frame = yolo_results[0].plot()

            gesture_text = "No Gesture Detected"
            if len(yolo_results[0].boxes) > 0:
                best_box_idx = np.argmax(yolo_results[0].boxes.conf.cpu().numpy())
                best_box = yolo_results[0].boxes[best_box_idx]
                class_id = int(best_box.cls[0])
                confidence = float(best_box.conf[0])
                gesture_name = yolo_model.names[class_id]
                gesture_text = f"{gesture_name} ({confidence*100:.1f}%)"

            cv2.putText(annotated_frame, f"Gesture: {gesture_text}", (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
            rgb.flags.writeable = False
            results = pose.process(rgb)
            rgb.flags.writeable = True

            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark
                left_shoulder = landmark_to_world(landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER])
                right_shoulder = landmark_to_world(landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER])
                left_elbow = landmark_to_world(landmarks[mp_pose.PoseLandmark.LEFT_ELBOW])
                right_elbow = landmark_to_world(landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW])
                left_wrist = landmark_to_world(landmarks[mp_pose.PoseLandmark.LEFT_WRIST])
                right_wrist = landmark_to_world(landmarks[mp_pose.PoseLandmark.RIGHT_WRIST])
                left_hip = landmark_to_world(landmarks[mp_pose.PoseLandmark.LEFT_HIP])
                right_hip = landmark_to_world(landmarks[mp_pose.PoseLandmark.RIGHT_HIP])
                left_knee = landmark_to_world(landmarks[mp_pose.PoseLandmark.LEFT_KNEE])
                right_knee = landmark_to_world(landmarks[mp_pose.PoseLandmark.RIGHT_KNEE])
                left_ankle = landmark_to_world(landmarks[mp_pose.PoseLandmark.LEFT_ANKLE])
                right_ankle = landmark_to_world(landmarks[mp_pose.PoseLandmark.RIGHT_ANKLE])

                left_shoulder = smooth_position("ls", left_shoulder)
                right_shoulder = smooth_position("rs", right_shoulder)
                left_elbow = smooth_position("le", left_elbow)
                right_elbow = smooth_position("re", right_elbow)
                left_wrist = smooth_position("lw", left_wrist)
                right_wrist = smooth_position("rw", right_wrist)
                left_hip = smooth_position("lh", left_hip)
                right_hip = smooth_position("rh", right_hip)
                left_knee = smooth_position("lk", left_knee)
                right_knee = smooth_position("rk", right_knee)
                left_ankle = smooth_position("la", left_ankle)
                right_ankle = smooth_position("ra", right_ankle)

                shoulder_center = (left_shoulder + right_shoulder) / 2.0
                hip_center = (left_hip + right_hip) / 2.0

                set_body_between(TORSO, hip_center, shoulder_center)

                head_position = shoulder_center + np.array([0, 0, 0.2])
                p.resetBasePositionAndOrientation(HEAD, head_position.tolist(), [0, 0, 0, 1])

                set_body_between(LEFT_UPPER_ARM, left_shoulder, left_elbow)
                set_body_between(LEFT_FOREARM, left_elbow, left_wrist)
                set_body_between(RIGHT_UPPER_ARM, right_shoulder, right_elbow)
                set_body_between(RIGHT_FOREARM, right_elbow, right_wrist)
                set_body_between(LEFT_THIGH, left_hip, left_knee)
                set_body_between(LEFT_SHIN, left_knee, left_ankle)
                set_body_between(RIGHT_THIGH, right_hip, right_knee)
                set_body_between(RIGHT_SHIN, right_knee, right_ankle)

                markers = {
                    LEFT_SHOULDER_MARKER: left_shoulder, RIGHT_SHOULDER_MARKER: right_shoulder,
                    LEFT_ELBOW_MARKER: left_elbow, RIGHT_ELBOW_MARKER: right_elbow,
                    LEFT_HIP_MARKER: left_hip, RIGHT_HIP_MARKER: right_hip,
                    LEFT_KNEE_MARKER: left_knee, RIGHT_KNEE_MARKER: right_knee
                }
                for mid, pos in markers.items():
                    p.resetBasePositionAndOrientation(mid, pos.tolist(), [0, 0, 0, 1])

                mp_drawing.draw_landmarks(annotated_frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                         mp_drawing.DrawingSpec(color=(0, 255, 255), thickness=2, circle_radius=3),
                                         mp_drawing.DrawingSpec(color=(255, 255, 255), thickness=2))

            pybullet_frame = capture_pybullet_frame()
            pybullet_frame = cv2.resize(pybullet_frame, (annotated_frame.shape[1] // 2, annotated_frame.shape[0] // 2))
            frame_resized = cv2.resize(annotated_frame, (annotated_frame.shape[1] // 2, annotated_frame.shape[0] // 2))

            combined_frame = np.hstack((frame_resized, pybullet_frame))

            cv2.putText(combined_frame, f"YOLO + MediaPipe (Frame {i+1}/{FRAMES_TO_RUN})", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            cv2.putText(combined_frame, "PyBullet Avatar", (frame_resized.shape[1] + 10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            p.stepSimulation()

            clear_output(wait=True)
            cv2_imshow(combined_frame)
            time.sleep(0.05)

        except Exception as e:
            print(f"Error: {e}")
            break

p.disconnect()